In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
# 改了 src/ 里的 .py，notebook 不用重启就能用上新代码
from signal_mysql import database as mysql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, db, geo, privacy, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT, AREA_COLOR, SEG_COLOR  # 配色常量
viz.apply_theme()                           # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)       # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                   # 分析日期（口径常量）
NB = "D11_工程化重构"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"    # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
from signal_mysql import database as mysql
import shutil
import tempfile
from pathlib import Path
from signal_insight import dims, dwd, dws
REPLAY = mysql.sandbox_schema("d11_replay")          # 在副本库上重演：不碰你自己的 signal_insight
_drop = mysql.connect(REPLAY); _drop.execute(f"DROP DATABASE IF EXISTS {mysql.ident(REPLAY)}"); _drop.commit(); _drop.close()
mysql.create_schema(REPLAY)
mysql.engine(mysql.settings_url(REPLAY)).dispose()   # 丢掉指向被删库的旧连接池
src, dst = mysql.connect(config.DB_SCHEMA), mysql.connect(REPLAY)
try:
    for name in mysql.table_names(src, "ods_"):
        dst.execute(f"CREATE TABLE {mysql.ident(name)} LIKE {mysql.ident(config.DB_SCHEMA)}.{mysql.ident(name)}")
        dst.execute(f"INSERT INTO {mysql.ident(name)} SELECT * FROM {mysql.ident(config.DB_SCHEMA)}.{mysql.ident(name)}")
    dst.commit()
finally:
    src.close(); dst.close()

def run_day_v0_2(day, path):
    """v0.2 的 run_day 骨架。两处要命的写法：① 汇总层整表替换；② 热点中心按“当天”的数据重算。"""
    settings = config.load_settings()
    con = db.connect(path)
    clean, _ = dwd.clean_signal(mysql.read_frame("SELECT * FROM ods_signal", con), day)
    db.write_partition(con, clean, "dwd_signal", day)                     # 明细层 D05 起就按日期分区：它没出事
    districts = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_district_boundary", con), "district")
    aois = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", con), "name")
    dim_cell = dims.build_dim_cell(mysql.read_frame("SELECT * FROM ods_cell", con), districts, aois)
    h = settings["hotspot"]
    hotspot = dims.build_dim_hotspot(clean, dim_cell, h["top_n"], h["name"], h["radius_m"])    # ② 每天重算
    db.replace_table(con, hotspot, "dim_hotspot")
    sig = clean.merge(dim_cell[["lac_cell", "site_id", "lng", "lat", "district"]], on="lac_cell")
    stay = dws.build_stay(sig, settings, districts, aois, hotspot.iloc[0])
    night, trip = dws.build_night(stay, day, settings), dws.build_trip(stay, settings)
    for table, df in [("dws_stay", stay), ("dws_night", night), ("dws_trip", trip),
                      ("dws_user_day", dws.build_user_day(sig, stay, trip, night, day, settings))]:
        db.replace_table(con, df, table)                                   # ① 整表替换：只有一天时没问题
    con.close()

def snapshot(path):
    con = db.connect(path)
    q = lambda sql: con.execute(sql).fetchone()[0]
    out = {"过夜地（dws_night 行数）": q("SELECT COUNT(*) FROM dws_night"),
           "停留（dws_stay 行数）": q("SELECT COUNT(*) FROM dws_stay"),
           "傍晚访客": q("SELECT COUNT(*) FROM dws_user_day WHERE is_evening_visitor"),
           "出行（dws_trip 行数）": q("SELECT COUNT(*) FROM dws_trip"),
           "西塔中心经度": q("SELECT lng FROM dim_hotspot"), "西塔中心纬度": q("SELECT lat FROM dim_hotspot")}
    con.close()
    return out

run_day_v0_2(DAY, REPLAY)
before = snapshot(REPLAY)
run_day_v0_2("2018-10-02", REPLAY)                                          # 客户说：“把 10-02 也跑一下”
after = snapshot(REPLAY)
incident = pd.DataFrame({"先跑 10-03": before, "再跑 10-02 之后": after}).iloc[:4].astype(int)
d_m = geo.haversine(before["西塔中心纬度"], before["西塔中心经度"], after["西塔中心纬度"], after["西塔中心经度"])
print(f"西塔中心漂移了 {d_m:.0f} 米；没有任何报错")
incident

In [ ]:
# ── 出图 D11_01：哑铃图 ─────────────────────────────────────────────
# 为什么用哑铃图：要表达的是“同一个指标，前后两个值”，线的长度就是变化量。
# 蓝点 = 之前、红点 = 之后：红色在这里是警示（结果被破坏了）。几个指标从几十到几万，用对数轴。
fig, ax = plt.subplots(figsize=(10, 4.2))
y = np.arange(len(incident))[::-1]
for yi, (name, r) in zip(y, incident.iterrows()):
    a, b = r["先跑 10-03"], r["再跑 10-02 之后"]
    ax.plot([a, b], [yi, yi], color=GRAY, lw=2, zorder=1)
    ax.scatter([a], [yi], s=80, color=MAIN, zorder=2)
    ax.scatter([b], [yi], s=80, color=RED, zorder=2)
    ax.text(max(a, b) * 1.3, yi, f"{a:,.0f} → {b:,.0f}", va="center", fontsize=10.5)
ax.set_xscale("log"); ax.set_xlim(1, 3e5)
ax.set_yticks(y, incident.index); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
ax.set_xticks([1, 10, 100, 1000, 10000, 100000], ["1", "10", "100", "1,000", "1 万", "10 万"])
ax.text(1.2, y[0] + 0.45, "● 先跑 10-03", color=MAIN, fontsize=10)
ax.text(12, y[0] + 0.45, "● 再跑 10-02 之后", color=RED, fontsize=10)
viz.finish(fig, f"多跑一天，10-03 的过夜地表从 {before['过夜地（dws_night 行数）']:,} 行变成 {after['过夜地（dws_night 行数）']:,} 行，没有任何报错",
           f"同一个库，用 v0.2 的流水线先跑 10-03、再跑 10-02；横轴对数刻度。西塔中心同时漂移了约 {d_m:.0f} 米（热点按当天数据重算）；\n"
           "10-02 的数据量只有 10-03 的 2.8%，却没有任何检查拦住它",
           SOURCE, "D11_01_事故复盘.png")

In [ ]:
from signal_mysql import database as mysql
settings = config.load_settings()
hot_cfg = dims.dim_hotspot_from_settings(settings).iloc[0]
hot_d04 = mysql.read_frame("SELECT * FROM dim_hotspot", con).iloc[0]
assert (hot_cfg["lng"], hot_cfg["lat"]) == (hot_d04["lng"], hot_d04["lat"])
print(f"配置版本 {settings['version']}；热点中心 ({hot_cfg['lng']:.6f}, {hot_cfg['lat']:.6f})，与 D04 算出的完全相同")

In [ ]:
from signal_mysql import database as mysql
def make_demo():
    demo = mysql.connect(mysql.sandbox_schema("notebook_demo"))
    mysql.write_frame(pd.DataFrame({"dt": ["D1"] * 3 + ["D2"] * 3, "v": range(6)}), "t", demo, if_exists="replace", index=False)
    return demo

def counts(demo):
    return mysql.read_frame("SELECT dt, COUNT(*) AS n FROM t GROUP BY dt", demo).set_index("dt")["n"].to_dict()

bad = pd.DataFrame({"v": [{"嵌套": 1}]})     # 上游把字段改成了嵌套结构：数据库存不了，插入时一定会出错

# 错误示范：D05 的写法，删完就提交
demo = make_demo()
demo.execute("DELETE FROM t WHERE dt = 'D2'"); demo.commit()
try:
    mysql.write_frame(bad.assign(dt="D2"), "t", demo, if_exists="append", index=False)
except Exception as e:
    print("写入出错：", type(e).__name__)
print("两次提交： ", counts(demo))

# D11 的写法：同一个事务
demo = make_demo()
try:
    db.write_partition(demo, bad, "t", "D2")
except Exception as e:
    print("写入出错：", type(e).__name__)
print("同一个事务：", counts(demo))

In [ ]:
# 对账：同一个事务里，写入失败时 D2 的旧数据还在
assert counts(demo) == {"D1": 3, "D2": 3}
print("✅ 对账通过：写入失败整体回滚，D2 的 3 行旧数据原样还在")

In [ ]:
from signal_insight import __main__ as cli_main, ingest, pipeline

def cli(*args):
    """和在终端里敲 python -m signal_insight … 一样，只是在 notebook 里直接调用。"""
    return cli_main.main(list(args))

landing = next((p for p in [config.LANDING_DIR / "ods_signal_2018-10-04.parquet",
                            config.LANDING_DIR / "ods_signal_2018-10-04.db"] if p.exists()),
               config.LANDING_DIR / "ods_signal_2018-10-04.db")
assert landing.exists(), f"找不到 {landing}：把讲师发的 ods_signal_2018-10-04 分区文件放进 data/landing/"
cli("init")
cli("run", "--date", DAY)

In [ ]:
wh = db.connect(config.WAREHOUSE_SCHEMA)
# 入库还不是幂等的：同一个文件入两次，ODS 会多出一整批重复数据（D12 用文件指纹解决）。今天先自己查一下。
if "2018-10-04" not in ingest.ods_dates(wh).index:
    cli("ingest", str(landing))
cli("run", "--date", "2018-10-04")

In [ ]:
# 对账：跑完 10-04，生产库里 10-03 一行没变；开发库拒绝 10-02
q = lambda sql: wh.execute(sql).fetchone()[0]
n_stay = q("SELECT COUNT(*) FROM dws_stay WHERE dt = '2018-10-03'")
n_ev = q("SELECT COUNT(*) FROM dws_user_day WHERE dt = '2018-10-03' AND is_evening_visitor")
n_night = q("SELECT COUNT(*) FROM dws_night WHERE dt = '2018-10-03'")
assert (n_stay, n_ev, n_night) == (19_015, 3_310, 3_607)
assert q("SELECT COUNT(*) FROM dws_stay WHERE dt = '2018-10-04'") > 0
try:
    pipeline.run_day("2018-10-02")
    raise AssertionError("开发库应该拒绝 10-02")
except ValueError as e:
    print("开发库：", str(e).split("：")[0])
print(f"✅ 对账通过：生产库 10-03 停留 {n_stay:,}、傍晚访客 {n_ev:,}、过夜地 {n_night:,}，跑完 10-04 一行没变")

In [ ]:
from signal_mysql import database as mysql
import yaml
print(mysql.read_frame("SELECT dt, run_id, config_version, COUNT(*) AS n_steps, ROUND(SUM(elapsed_sec), 1) AS total_sec "
                       "FROM etl_log GROUP BY dt, run_id ORDER BY MAX(run_at) DESC LIMIT 3", wh).to_string(index=False))
text = config.SETTINGS_PATH.read_text(encoding="utf-8")
same = yaml.safe_load("# 只是加一行注释\n" + text)
changed = yaml.safe_load(text.replace("min_minutes: 30", "min_minutes: 20", 1))
print("原配置：      ", config.config_hash(settings))
print("加一行注释：  ", config.config_hash(same))
print("改最短时长：  ", config.config_hash(changed))

In [ ]:
from signal_mysql import database as mysql
# ── 出图 D11_03：生产库上处理 10-03 每一步的耗时 ─────────────────────────
# 为什么只取“读、算”的步骤画：写库步骤很快，混在一起会把要看的那几根压扁；横向条形，最长的放最上面。
last = mysql.read_frame("SELECT * FROM etl_log WHERE dt = ? AND run_id = (SELECT run_id FROM etl_log WHERE dt = ? "
                        "ORDER BY run_at DESC LIMIT 1)", wh, params=(DAY, DAY))
t = last.groupby("step")["elapsed_sec"].sum().sort_values()
fig, ax = plt.subplots(figsize=(10, max(4.6, 0.3 * len(t) + 2)))   # D13：步骤变多时图跟着变高
colors = [MAIN if s == t.idxmax() else GRAY for s in t.index]
ax.barh(t.index, t.values, color=colors)
for i, v in enumerate(t.values):
    ax.text(v + t.max() * 0.01, i, f"{v:.2f} 秒", va="center", fontsize=9.5)
ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.set_xlabel("秒")
viz.finish(fig, f"处理一天共 {t.sum():.1f} 秒，最慢的一步是 {t.idxmax()}（{t.max() / t.sum():.0%}）",
           f"生产库、{DAY}、最近一次运行（批次 {last['run_id'].iloc[0]}，配置版本 {last['config_version'].iloc[0]}）；数据来自 etl_log",
           SOURCE, "D11_03_各步耗时.png")

In [ ]:
import logging
import sys
logger = logging.getLogger("signal_insight.demo")
logger.handlers.clear()
h = logging.StreamHandler(sys.stdout)
h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(name)s | %(message)s", "%H:%M:%S"))
logger.addHandler(h)
logger.setLevel(logging.INFO)
logger.debug("级别低于 INFO，不输出")
logger.info("开始处理分区 %s", DAY)
logger.warning("分区 %s 数据量只有基准的 %.1f%%", "2018-10-02", 2.8)

In [ ]:
from signal_mysql import database as mysql
from signal_insight.viz.lint import lint_figure
uh = mysql.read_frame("SELECT hour, COUNT(*) AS users FROM dws_user_hour WHERE dt = ? AND dist_xita_m <= 500 GROUP BY hour", wh, params=(DAY,))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
ax1.bar(uh["hour"], uh["users"], color=MAIN)
ax1b = ax1.twinx(); ax1b.plot(uh["hour"], uh["users"].cumsum(), color=RED)     # 错 1：双轴
ax2.pie(uh["users"].head(8), labels=[f"{h} 点" for h in uh["hour"].head(8)])   # 错 2：8 块的饼图
fig.suptitle("西塔分时人数")                                                   # 错 3、4：标题不是结论，没有口径和来源
problems = lint_figure(fig)
plt.close(fig)
problems

In [ ]:
# ── 出图 D11_04：同样的数据，按规范画 ───────────────────────────────────
# 一张图只讲一件事：每小时在西塔 500 米内的人数；峰值用强调色，其余灰色；标题写结论、带数字。
fig, ax = plt.subplots(figsize=(10, 4.2))
peak = uh.loc[uh["users"].idxmax()]
ax.bar(uh["hour"], uh["users"], color=[MAIN if h == peak["hour"] else GRAY for h in uh["hour"]], width=0.75)
ax.set_xticks(range(0, 24, 3), [f"{h} 点" for h in range(0, 24, 3)]); ax.set_ylabel("人")
viz.finish(fig, f"西塔 {int(peak['hour'])} 点在场 {int(peak['users']):,} 人，是全天高峰",
           f"生产库 {DAY}；每人每小时一个位置（dws_user_hour），离西塔中心 500 米以内计为在场",
           SOURCE, "D11_04_规范图示例.png")
print("规范检查：", lint_figure(fig) or "通过")

In [ ]:
# 对账：坏图被挑出至少 3 类问题；按规范画的图通过
assert any("双轴" in p for p in problems) and any("饼图" in p for p in problems) and any("viz.finish" in p for p in problems)
assert lint_figure(fig) == []
print(f"✅ 对账通过：坏图查出 {len(problems)} 处问题，规范图通过")

In [ ]:
import ast
import subprocess
import sys
PKG = config.PROJECT_ROOT / "src" / "signal_insight"
rows = []
for p in sorted(PKG.rglob("*.py")):
    tree = ast.parse(p.read_text(encoding="utf-8"))
    doc = (ast.get_docstring(tree) or "").split("\n")[0]
    defs = [n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.ClassDef))]
    rows.append({"模块": str(p.relative_to(PKG)), "行数": len(p.read_text(encoding="utf-8").splitlines()),
                 "函数": len(defs), "职责": doc[:40]})
print(subprocess.run([sys.executable, str(config.PROJECT_ROOT / "tools" / "check_deps.py")],
                     capture_output=True, text=True, encoding="utf-8").stdout)
pd.DataFrame(rows)

In [ ]:
import importlib.util
# ── 出图 D11_02：分层依赖图 ───────────────────────────────────────────
# 为什么按层摆成几列、让箭头全部朝左：**位置本身就是结论**，任何一个朝右的箭头一眼就能看出来。
# 不用自动布局：自动布局每次位置都不一样，还会把“分层”这个最重要的信息打乱。图要表达什么，就手工摆成什么样。
spec = importlib.util.spec_from_file_location("check_deps", config.PROJECT_ROOT / "tools" / "check_deps.py")
check_deps = importlib.util.module_from_spec(spec); spec.loader.exec_module(check_deps)
graph, problems_dep = check_deps.check()
edges = [(a, b) for a, deps in graph.items() for b in deps]
names = {0: "最底层", 1: "通用能力", 2: "计算", 3: "接入", 4: "编排", 5: "入口"}
cols = {}
for m, lv in check_deps.LAYERS.items():
    cols.setdefault(lv, []).append(m)
pos = {m: (lv, (len(ms) - 1) / 2 - j) for lv, ms in cols.items() for j, m in enumerate(ms)}
fig, ax = plt.subplots(figsize=(11, 4.8))
for a, b in edges:
    (x1, y1), (x2, y2) = pos[a], pos[b]
    ax.annotate("", (x2 + 0.22, y2), (x1 - 0.22, y1),
                arrowprops=dict(arrowstyle="-|>", color=GRAY, lw=0.9, shrinkA=4, shrinkB=4, connectionstyle="arc3,rad=0.12"))
for lv, ms in cols.items():
    ax.text(lv, 2.9, names[lv], ha="center", fontsize=11, fontweight="bold", color=MUTED)
    for m in ms:
        x, yy = pos[m]
        dark = lv in (2, 4)
        ax.text(x, yy, m, ha="center", va="center", fontsize=10.5, color="white" if dark else INK,
                bbox=dict(boxstyle="round,pad=0.45", fc=MAIN if dark else MAIN_LIGHT, ec="none"))
ax.set_xlim(-0.6, 5.6); ax.set_ylim(-3.1, 3.3); ax.axis("off")
viz.finish(fig, f"{len(edges)} 条依赖全部从右指向左，没有循环：底层模块不知道谁在用它",
           "箭头 = “导入了”；计算模块（dwd、dims、dws）只做 DataFrame 进、DataFrame 出，读写数据库只在 db、ingest、pipeline；\n"
           "所以计算模块能单独测试、能在 notebook 里单独调用，换数据库只改 db",
           SOURCE, "D11_02_模块依赖.png")

In [ ]:
# 对账：依赖检查无违规、无环
assert problems_dep == []
print(f"✅ 对账通过：{len(edges)} 条依赖，全部从上往下，没有环")

In [ ]:
import shutil
import subprocess
import tempfile
from pathlib import Path
repo = Path(tempfile.mkdtemp())
shutil.copy(config.PROJECT_ROOT / ".gitignore", repo / ".gitignore")
for f in ["data/signal_warehouse", "data/landing/ods_signal_2018-10-04.db",
          "docs/指标口径表.csv", "src/signal_insight/ingest.py"]:
    (repo / f).parent.mkdir(parents=True, exist_ok=True); (repo / f).write_text("x")
subprocess.run(["git", "init", "-q"], cwd=repo)
out = subprocess.run(["git", "check-ignore", "-v", "data/signal_warehouse", "data/landing/ods_signal_2018-10-04.db",
                      "docs/指标口径表.csv", "src/signal_insight/ingest.py"],
                     cwd=repo, capture_output=True, text=True, encoding="utf-8").stdout
print(out or "（没有任何文件被挡住）")

In [ ]:
from signal_mysql import database as mysql
import cProfile
import pstats
sig = mysql.read_frame("SELECT s.imsi, s.ts, s.lac_cell, c.lng, c.lat FROM dwd_signal s JOIN dim_cell c ON s.lac_cell = c.lac_cell "
                       "WHERE s.dt = ?", wh, params=(DAY,), parse_dates=["ts"])
districts = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_district_boundary", wh), "district")
aois = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", wh), "name")
hot = mysql.read_frame("SELECT * FROM dim_hotspot", wh).iloc[0]
prof = cProfile.Profile()
prof.enable()
stay_prof = dws.build_stay(sig, settings, districts, aois, hot)
prof.disable()
stats = pstats.Stats(prof).sort_stats("cumulative")
rows = [{"函数": f"{fn[2]}（{Path(fn[0]).name}）", "调用次数": st[1], "累计秒": round(st[3], 2)}
        for fn, st in stats.stats.items() if "signal_insight" in fn[0]]
pd.DataFrame(rows).sort_values("累计秒", ascending=False).head(6)

In [ ]:
BRANCH = (ast.If, ast.For, ast.While, ast.Try, ast.BoolOp, ast.IfExp, ast.comprehension)
rows = []
for p in PKG.rglob("*.py"):
    for n in ast.walk(ast.parse(p.read_text(encoding="utf-8"))):
        if isinstance(n, ast.FunctionDef):
            rows.append({"函数": f"{p.stem}.{n.name}", "行数": n.end_lineno - n.lineno + 1,
                         "分支数": sum(isinstance(x, BRANCH) for x in ast.walk(n)), "参数个数": len(n.args.args)})
pd.DataFrame(rows).sort_values(["分支数", "行数"], ascending=False).head(8)

In [ ]:
# 今日对账（交付前最后跑一遍）
assert (n_stay, n_ev, n_night) == (19_015, 3_310, 3_607) and problems_dep == [] and lint_figure(fig) == []
print("✅ D11 对账全部通过：事故重演、事务回滚、生产库两天互不覆盖、依赖无环、规范图通过")
wh.close(); con.close()